# D1 · Reading `cv_results_`: who really wins, and which parameter matters

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Analyze the results table of a `GridSearchCV` on a random forest: separate the truly better combinations from the
equivalent ones, apply the standard error rule to choose the simplest model and measure overfitting and the importance
of each hyperparameter.

## Dataset

`ds.loans(with_missing=False)` (four numeric columns and the target `defaulted`). The setup cell runs a `GridSearchCV`
on a `RandomForestClassifier(n_estimators=60)` with `max_depth` in `[2, 4, 6, 10, None]` and `min_samples_leaf` in `[1, 5, 20]`
(15 combinations, 5 folds, `return_train_score=True`) and produces `r`, the DataFrame of `cv_results_`.

## Educational purpose

Stop looking only at `best_params_`. A results table contains much more: how clear the win is, how much of the
difference is within the noise, where the model memorizes and which hyperparameter really moves the performance.

## Task

1. Sort `r` by `rank_test_score`. Save `best` (the first row), `best_mean` and `best_std` (`mean_test_score` and `std_test_score`).
2. Apply the **standard error rule**: save in `candidates_1se` (DataFrame) the rows with `mean_test_score >= best_mean − best_std`, and in `n_candidates_1se` their number. They are the combinations statistically indistinguishable from the best.
3. Among the `candidates_1se` choose the **simplest**: minimum `max_depth` (with `None` considered the maximum); on a tie, larger `min_samples_leaf`. Save the row in `choice_1se`.
4. Compute `gap = mean_train_score − mean_test_score` for every row and save `best_gap` (of the best one) and `max_gap` (the largest in the table).
5. With `groupby` on the values of each hyperparameter (mean of `mean_test_score`), save `depth_range` and `leaf_range` (maximum minus minimum of the mean per value) and `most_influential_parameter` (`"max_depth"` or `"min_samples_leaf"`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]]
y = df["defaulted"]
grid = {"max_depth": [2, 4, 6, 10, None], "min_samples_leaf": [1, 5, 20]}
search = GridSearchCV(RandomForestClassifier(n_estimators=60, random_state=0), grid,
                      cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc", return_train_score=True).fit(X, y)
r = pd.DataFrame(search.cv_results_)
r["max_depth"] = r["param_max_depth"].map(lambda v: 99 if v is None else v).astype(int)       # None = no limit
r["min_samples_leaf"] = r["param_min_samples_leaf"].astype(int)
r[["max_depth", "min_samples_leaf", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]].sort_values("rank_test_score").head(5).round(3)

## Your work

In [ ]:
r = r.sort_values("rank_test_score").reset_index(drop=True)
best = ...
best_mean = ...
best_std = ...

In [ ]:
candidates_1se = ...
n_candidates_1se = ...
choice_1se = ...

In [ ]:
r["gap"] = ...
best_gap = ...
max_gap = ...

In [ ]:
depth_range = ...
leaf_range = ...
most_influential_parameter = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For the choice: `candidates_1se.sort_values(["max_depth", "min_samples_leaf"], ascending=[True, False]).iloc[0]`. Range: `m = r.groupby("max_depth")["mean_test_score"].mean(); m.max() - m.min()`.'):
    check(best["rank_test_score"] == 1 and close(best_mean, r["mean_test_score"].max()) and best_std > 0,
          "the best is the first row and has a positive standard deviation", "best = r.iloc[0]")
    expected = r[r["mean_test_score"] >= best_mean - best_std]
    check(n_candidates_1se == len(expected) and n_candidates_1se >= 5,
          "many combinations are statistically indistinguishable from the best (at least 5)", "candidates_1se = rows with mean_test_score >= best_mean - best_std")
    order = candidates_1se.assign(_d=candidates_1se["max_depth"]).sort_values(["_d", "min_samples_leaf"], ascending=[True, False])
    check(choice_1se["max_depth"] == order.iloc[0]["max_depth"] and choice_1se["min_samples_leaf"] == order.iloc[0]["min_samples_leaf"],
          "choice_1se: the simplest combination among the equivalent ones", "minimum max_depth (None = 99), on a tie maximum min_samples_leaf")
    check(choice_1se["max_depth"] <= best["max_depth"] and choice_1se["mean_test_score"] >= best_mean - best_std,
          "the choice is as simple as the best or more so, without losing appreciably", "the 1-SE choice cannot be more complex than the best")
    check(close(best_gap, best["mean_train_score"] - best["mean_test_score"]) and max_gap >= best_gap and max_gap > 0.2,
          "the maximum gap (overfitting) exceeds 0.2", "gap = mean_train_score - mean_test_score")
    check(most_influential_parameter in ("max_depth", "min_samples_leaf") and
          most_influential_parameter == ("max_depth" if depth_range > leaf_range else "min_samples_leaf"),
          "most_influential_parameter consistent with the ranges", "the parameter with the larger range of the means")
done()

---
The commented solution is in `solutions/D_interpret/D1_reading_cv_results.ipynb`. Open it after you pass the check, or after you have really tried.